# Heimdall: can a bank spot mule accounts, and hold their money, before it flags them itself?

**The question:** *On 1 January 2025, using only what the bank knew before that date, which accounts will it flag as mules during 2025, and how much of the money flowing into them could have been held?*

This is an **out-of-time** test, the way a bank would really use a model:
- the methods **learn only from the past** (account snapshots from 2022 to 2024, labelled by what the bank flagged in the following 12 months);
- they are **tested once, on the future** (accounts as they looked on 1 Jan 2025, labelled by what the bank flagged during 2025);
- we then **replay January–June 2025** month by month and measure the **share of mule money that would have been held** before the bank's own flag, and how many genuine customers would have had a hold.

- **Data:** RBIH × IIT Delhi National Fraud Prevention Challenge (NFPC) Phase-2 dataset, Kaggle `abhyudayrbih/rbih-nfpc-phase-2`, licence CC BY-NC-SA 4.0 (non-commercial). The data is **synthetic**: results describe this dataset, not a real bank.
- **How to run:** attach the dataset, set *Accelerator* to *None*, then **Run All**. The full run takes roughly 25–35 minutes. For a quick trial, set `N_FILES = 20` in the next cell first (the numbers will then be meaningless, but every step runs).
- The steps follow Figure 2 in the project README.

## Settings

In [ ]:
import glob, os, time, json
import multiprocessing as mp
import numpy as np
import pandas as pd

TEST_DATE = pd.Timestamp('2025-01-01')                                # "today" for the test
TEST_LABEL_END = pd.Timestamp('2026-01-01')                           # test label: flagged during 2025
TRAIN_DATES = pd.date_range('2022-01-01', '2024-01-01', freq='6MS')   # learning snapshots; each 12-month outcome is known by 1 Jan 2025
REFRESH = pd.date_range('2025-01-01', '2025-06-01', freq='MS')        # money replay: re-score at the start of each month
N_FILES = None                 # None = all 396 files; e.g. 20 for a quick trial run
WORKERS = 4                    # Kaggle CPU notebooks have 4 cores
INPUT_ROOT = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'

## Step 0. Find the data and read the labels
Kaggle puts attached datasets under `/kaggle/input`. We search for the labels file so the notebook works whatever the folder is called.
`mule_flag_date` is the day the bank flagged the account. It tells us **when** each mule became known, which is what makes an honest past/future test possible.

In [ ]:
DATA = os.path.dirname(glob.glob(INPUT_ROOT + '/**/train_labels.parquet', recursive=True)[0])
parts = sorted(glob.glob(DATA + '/transactions/batch-*/part_*.parquet'))
if N_FILES: parts = parts[:N_FILES]
print('data folder:', DATA)
print(len(parts), 'transaction files')          # 396 for the full dataset

lab = pd.read_parquet(DATA + '/train_labels.parquet', columns=['account_id', 'is_mule', 'mule_flag_date'])
lab['acct'] = lab['account_id'].str[5:].astype('int32')                     # 'ACCT_000123' -> 123 (smaller and faster)
lab['flag'] = pd.to_datetime(lab['mule_flag_date'], errors='coerce')
lab = lab.set_index('acct')[['is_mule', 'flag']]
print(len(lab), 'labelled accounts,', int(lab['is_mule'].sum()), 'mules')
print('mules flagged per year:'); print(lab.loc[lab['is_mule'] == 1, 'flag'].dt.year.value_counts().sort_index())

## Look at one file before doing anything
`txn_type`: **C** = money in (credit), **D** = money out (debit). `channel`: **UPC/UPD** = UPI credit/debit.

In [ ]:
one = pd.read_parquet(parts[0])
print(one.shape)
print(one['txn_type'].value_counts())
print(one['channel'].value_counts().head(5))
one.head()

## Step 1. Summarise every file into three small tables, then join the pieces of each account
400 million transactions are too many to re-read for every date we test. So we read each file **once** and keep three small tables that let us rebuild any account **as it looked on any date**:

- **`M`**, one row per *(account, month)*: number of transactions, money in, money out, UPI transactions, credits of Rs 1,000 or more, first and last transaction time.
- **`E`**, one row per *(account, contact)*: the first time money came **in** from that contact and the first time money went **out** to them.
- **`K`**, every credit of Rs 1,000 or more into an account that the bank flagged as a mule in 2025 or later, between January and June 2025 (used for the money replay).

`BANK_INTEREST` rows are the bank's own interest postings, not customer behaviour, so we drop them.
Some accounts are spread over two or three files (about 7% of accounts, and about half of the mules), so after summarising we **add the pieces together per account**. Skipping this step makes those accounts look smaller than they are.

In [ ]:
COLS = ['account_id', 'transaction_timestamp', 'channel', 'amount', 'txn_type', 'counterparty_id']
raw = pd.read_parquet(DATA + '/train_labels.parquet', columns=['account_id', 'is_mule', 'mule_flag_date'])
TEST_MULES = set(raw['account_id'][(raw['is_mule'] == 1) & (pd.to_datetime(raw['mule_flag_date'], errors='coerce') >= TEST_DATE)])

def summarise_file(path):
    t = pd.read_parquet(path, columns=COLS)
    t = t[t['counterparty_id'] != 'BANK_INTEREST']
    t['ts'] = pd.to_datetime(t['transaction_timestamp'], format='ISO8601')        # some files use a space instead of 'T'
    t['acct'] = t['account_id'].str[5:].astype('int32')
    t['amt'] = t['amount'].abs()
    t['is_in'] = t['txn_type'] == 'C'
    t['month'] = ((t['ts'].dt.year - 2020) * 12 + t['ts'].dt.month - 1).astype('int16')
    t['money_in'] = t['amt'].where(t['is_in'], 0.0)
    t['money_out'] = t['amt'].where(~t['is_in'], 0.0)
    t['upi'] = t['channel'].isin(['UPC', 'UPD'])
    big = t['is_in'] & (t['amt'] >= 1000)
    t['big_in'] = t['amt'].where(big, 0.0); t['n_big_in'] = big
    M = t.groupby(['acct', 'month']).agg(n_txn=('amt', 'size'), money_in=('money_in', 'sum'), money_out=('money_out', 'sum'),
                                         n_upi=('upi', 'sum'), big_in=('big_in', 'sum'), n_big_in=('n_big_in', 'sum'),
                                         first_ts=('ts', 'min'), last_ts=('ts', 'max')).reset_index()
    t['ts_in'] = t['ts'].where(t['is_in']); t['ts_out'] = t['ts'].where(~t['is_in'])
    E = t.groupby(['acct', 'counterparty_id']).agg(first_in=('ts_in', 'min'), first_out=('ts_out', 'min')).reset_index()
    K = t[big & t['account_id'].isin(TEST_MULES) & (t['ts'] >= TEST_DATE) & (t['ts'] < TEST_DATE + pd.DateOffset(months=6))][['acct', 'ts', 'amt']]
    return M, E, K

start = time.time(); Ms, Es, Ks = [], [], []
with mp.get_context('fork').Pool(WORKERS) as pool:
    for i, (m_, e_, k_) in enumerate(pool.imap(summarise_file, parts)):     # imap keeps the file order, so results are repeatable
        Ms.append(m_); Es.append(e_); Ks.append(k_)
        if i % 50 == 0: print(i, 'files done,', round(time.time() - start), 'seconds')
M = pd.concat(Ms); E = pd.concat(Es); K = pd.concat(Ks); del Ms, Es, Ks
print('account-months that appear in more than one file:', M.duplicated(['acct', 'month']).sum())
M = M.groupby(['acct', 'month'], as_index=False).agg({'n_txn': 'sum', 'money_in': 'sum', 'money_out': 'sum', 'n_upi': 'sum',
                                                      'big_in': 'sum', 'n_big_in': 'sum', 'first_ts': 'min', 'last_ts': 'max'})
E = E.groupby(['acct', 'counterparty_id'], as_index=False).agg({'first_in': 'min', 'first_out': 'min'})
E['counterparty_id'] = E['counterparty_id'].astype('category')
E['first_any'] = E[['first_in', 'first_out']].min(axis=1)
print(f'{len(M):,} account-months, {len(E):,} account-contact pairs, {len(K):,} big credits into 2025 mules; {time.time() - start:.0f} s')

## Step 2. Static information: branch, PIN code, address
Used for the **address links** (accounts registered at the same address) and for the **fairness check** (rural vs urban branch).
An address is only used from the date it was last updated: if it changed after the snapshot date, the bank did not know it yet, so we ignore it.
The mobile-number update dates in this dataset all fall in 2025, after our test date, so mobile numbers are not used.

In [ ]:
acc = pd.read_parquet(DATA + '/accounts.parquet', columns=['account_id', 'branch_code', 'rural_branch'])
acc['acct'] = acc['account_id'].str[5:].astype('int32')
link = pd.read_parquet(DATA + '/customer_account_linkage.parquet')
link['acct'] = link['account_id'].str[5:].astype('int32')
cus = pd.read_parquet(DATA + '/customers.parquet', columns=['customer_id', 'customer_pin'])
dem = pd.read_parquet(DATA + '/demographics.parquet', columns=['customer_id', 'address', 'address_last_update_date'])
dem['addr_known_from'] = pd.to_datetime(dem['address_last_update_date'], errors='coerce')
S = acc[['acct', 'branch_code', 'rural_branch']].merge(link[['acct', 'customer_id']], on='acct', how='left') \
    .merge(cus, on='customer_id', how='left').merge(dem[['customer_id', 'address', 'addr_known_from']], on='customer_id', how='left')
S['rural'] = S['rural_branch'] == 'Y'
S = S.drop_duplicates('acct').set_index('acct')
print(len(S), 'accounts;', round(S['rural'].mean() * 100, 1), '% at rural branches')

## Step 3. Turn each account into numbers (features), as of any date T
Everything below uses **only** transactions, contacts, addresses and flags dated **before T**. That is what lets us test honestly: the model never sees the future.

| Group | Feature | Plain meaning |
|---|---|---|
| **Own behaviour (1-hop)** | `n_txn`, `n_contacts` | number of transactions; number of different contacts |
| | `two_way_share` | share of contacts where money went **both** ways (genuine relationships tend to be two-way) |
| | `contacts_in`, `contacts_out` | contacts that sent money in / received money out |
| | `money_in`, `money_out`, `out_in_ratio` | money received, money sent, and sent ÷ received (a pass-through account sends on most of what it gets) |
| | `upi_share`, `active_days`, `days_since_last_txn`, `txn_per_day` | share of UPI transactions; days since first / latest transaction; activity rate |
| | `in_last_month`, `out_last_month`, `out_in_ratio_last_month`, `new_contacts_last_month` | the same, for the last calendar month before T |
| **2-hop links** (no labels needed) | `cp_mean_degree`, `cp_max_degree` | how many accounts use the same counterparties (average and largest) |
| | `two_hop_reach`, `rare_two_hop_reach`, `rare_cp_share` | how many other accounts it reaches through shared counterparties (all, and only through rarely used ones) |
| | `receivers_fan_in`, `senders_fan_out` | for the people it pays: how many accounts pay them; for the people paying it: how many accounts they pay |
| | `shared_address_count` | other accounts registered at the same address |
| **Suspect list** (the bank's own earlier flags) | `flagged_contacts`, `flagged_contacts_weighted` | already-flagged accounts that share a counterparty with it (weighted: a shared counterparty used by few accounts counts more) |
| | `near_flagged_rare` | 2 = shares a rarely used counterparty with a flagged account; 1 = two such steps away; 0 = neither |
| | `address_flagged` | already-flagged accounts at the same address |
| **Tested separately (risky)** | `pin_flagged_share`, `branch_flagged_share` | share of flagged accounts in the same PIN code / branch. These can punish whole neighbourhoods, so they are only an ablation |

The 2-hop and suspect-list numbers are the kind a bank would compute in a **nightly batch job** and cache on each account's risk record, so the real-time check still reads one record. No graph neural network is needed.
`rural` is kept **only** for the fairness check. It is **not** given to the models. `account_age_days` is deliberately **not** used: hundreds of accounts in this dataset were "opened" after their first transaction, a planted trap.

In [ ]:
RARE = 2            # a counterparty is "rare" if at most 2 accounts have used it so far

def month_index(ts):
    return (ts.year - 2020) * 12 + ts.month - 1

BASE = ['n_txn', 'n_contacts', 'two_way_share', 'contacts_in', 'contacts_out', 'money_in', 'money_out', 'out_in_ratio',
        'upi_share', 'active_days', 'days_since_last_txn', 'txn_per_day', 'in_last_month', 'out_last_month',
        'out_in_ratio_last_month', 'new_contacts_last_month']
GRAPH = ['cp_mean_degree', 'cp_max_degree', 'two_hop_reach', 'rare_cp_share', 'rare_two_hop_reach',
         'receivers_fan_in', 'senders_fan_out', 'shared_address_count']
SUSPECT = ['flagged_contacts', 'flagged_contacts_weighted', 'near_flagged_rare', 'address_flagged']
RISKY = ['pin_flagged_share', 'branch_flagged_share']

def features_at(T):
    T = pd.Timestamp(T); mT = month_index(T)
    m = M[M['month'] < mT]
    A = m.groupby('acct').agg(n_txn=('n_txn', 'sum'), money_in=('money_in', 'sum'), money_out=('money_out', 'sum'),
                              n_upi=('n_upi', 'sum'), first_ts=('first_ts', 'min'), last_ts=('last_ts', 'max'))
    last = m[m['month'] == mT - 1].set_index('acct')
    F = pd.DataFrame(index=A.index)
    # own behaviour
    e = E[E['first_any'] < T].copy()
    e['has_in'] = e['first_in'] < T; e['has_out'] = e['first_out'] < T
    e['two_way'] = e['has_in'] & e['has_out']
    e['new_last_month'] = e['first_any'] >= (T - pd.DateOffset(months=1))
    g = e.groupby('acct').agg(n_contacts=('two_way', 'size'), two_way=('two_way', 'sum'), contacts_in=('has_in', 'sum'),
                              contacts_out=('has_out', 'sum'), new_contacts_last_month=('new_last_month', 'sum'))
    F['n_txn'] = A['n_txn']
    F['n_contacts'] = g['n_contacts']
    F['two_way_share'] = g['two_way'] / g['n_contacts']
    F['contacts_in'] = g['contacts_in']; F['contacts_out'] = g['contacts_out']
    F['money_in'] = A['money_in']; F['money_out'] = A['money_out']
    F['out_in_ratio'] = A['money_out'] / (A['money_in'] + 1)
    F['upi_share'] = A['n_upi'] / A['n_txn']
    F['active_days'] = (T - A['first_ts']).dt.days
    F['days_since_last_txn'] = (T - A['last_ts']).dt.days
    F['txn_per_day'] = A['n_txn'] / F['active_days'].clip(lower=1)
    F['in_last_month'] = last['money_in'].reindex(F.index).fillna(0)
    F['out_last_month'] = last['money_out'].reindex(F.index).fillna(0)
    F['out_in_ratio_last_month'] = F['out_last_month'] / (F['in_last_month'] + 1)
    F['new_contacts_last_month'] = g['new_contacts_last_month']
    # 2-hop links: how widely are my counterparties shared, as of T
    cp = e.groupby('counterparty_id', observed=True).agg(deg=('acct', 'size'), fan_in=('has_out', 'sum'), fan_out=('has_in', 'sum'))
    e = e.join(cp, on='counterparty_id')
    e['rare'] = e['deg'] <= RARE
    e['reach'] = e['deg'] - 1
    e['rare_reach'] = e['reach'].where(e['rare'], 0)
    e['recv_fan_in'] = e['fan_in'].where(e['has_out'])
    e['send_fan_out'] = e['fan_out'].where(e['has_in'])
    h = e.groupby('acct').agg(cp_mean_degree=('deg', 'mean'), cp_max_degree=('deg', 'max'), two_hop_reach=('reach', 'sum'),
                              rare_cp_share=('rare', 'mean'), rare_two_hop_reach=('rare_reach', 'sum'),
                              receivers_fan_in=('recv_fan_in', 'mean'), senders_fan_out=('send_fan_out', 'mean'))
    F = F.join(h)
    s = S.reindex(F.index)
    addr = s['address'].where(s['addr_known_from'] < T)            # only addresses already on file before T
    F['shared_address_count'] = (addr.map(addr.value_counts()) - 1).fillna(0)
    # suspect list: accounts the bank had ALREADY flagged before T
    flagged = set(lab.index[(lab['is_mule'] == 1) & (lab['flag'] < T)])
    e['is_flagged'] = e['acct'].isin(flagged)
    cpf = e.groupby('counterparty_id', observed=True)['is_flagged'].sum()
    e['cp_flagged'] = e['counterparty_id'].map(cpf).astype(float) - e['is_flagged']
    e['cp_flagged_w'] = e['cp_flagged'] / e['deg']
    e['rare5'] = e['deg'] <= 5
    k = e.groupby('acct').agg(flagged_contacts=('cp_flagged', 'sum'), flagged_contacts_weighted=('cp_flagged_w', 'sum'))
    F = F.join(k)
    r = e[e['rare5']]
    d1 = set(r['acct'][r['cp_flagged'] > 0])
    cp_d1 = set(r['counterparty_id'][r['acct'].isin(d1)])
    d2 = set(r['acct'][r['counterparty_id'].isin(cp_d1)]) - d1
    F['near_flagged_rare'] = np.where(F.index.isin(d1), 2, np.where(F.index.isin(d2), 1, 0))
    fl_addr = addr[addr.index.isin(flagged)].value_counts()
    F['address_flagged'] = addr.map(fl_addr).fillna(0) - (addr.notna() & F.index.isin(flagged))
    # risky neighbourhood features (ablation only)
    for col, key in [('pin_flagged_share', 'customer_pin'), ('branch_flagged_share', 'branch_code')]:
        grp = S[key]; n = grp.map(grp.value_counts())
        nf = grp.map(grp[S.index.isin(flagged)].value_counts()).fillna(0)
        F[col] = ((nf - S.index.isin(flagged)) / (n - 1).clip(lower=1)).reindex(F.index)
    F['rural'] = s['rural'].fillna(False).astype(bool)
    F['flagged_before'] = F.index.isin(flagged)
    return F

features_at(TEST_DATE).describe().T

## Step 4. Past for learning, future for testing
- **Learning snapshots** (1 Jan and 1 Jul, 2022 to 1 Jan 2024): label **1** = flagged within the next 12 months. A mule flagged later is labelled 0 here, because on that date nobody knew yet; that is the real situation a bank is in.
- **Test snapshot** (1 Jan 2025): label **1** = flagged during 2025. Mules flagged in 2026 or later are left out (outside the 12-month horizon).
- In every snapshot, accounts the bank had **already flagged** are left out: finding them again proves nothing.
- **Money replay:** the test accounts are rebuilt at the start of every month from January to June 2025, using only what was known on that day.

In [ ]:
def snapshot(T, label_end):
    F = features_at(T); L = lab.reindex(F.index)
    keep = L['is_mule'].notna() & ~F['flagged_before']
    F = F[keep].copy(); L = L[keep]
    F['y'] = ((L['is_mule'] == 1) & (L['flag'] >= T) & (L['flag'] < label_end)).astype('int8')
    F['later_mule'] = (L['is_mule'] == 1) & (L['flag'] >= label_end)
    F['snap'] = T
    return F

train = pd.concat([snapshot(T, min(T + pd.DateOffset(months=12), TEST_DATE)) for T in TRAIN_DATES])
test = snapshot(TEST_DATE, TEST_LABEL_END)
n_2026 = int(test['later_mule'].sum())
print('test mules flagged in 2026 or later (left out):', n_2026)
test = test[~test['later_mule']]
refresh = {}
for T in REFRESH:
    R = features_at(T).reindex(test.index)
    R['flagged_by_now'] = R.index.isin(lab.index[(lab['is_mule'] == 1) & (lab['flag'] < T)])
    refresh[T] = R
Kt = K[K['acct'].isin(test.index[test['y'] == 1])].copy()
Kt['flag'] = lab['flag'].reindex(Kt['acct']).to_numpy()
Kt = Kt[Kt['ts'] < Kt['flag']]                                              # only money that arrived BEFORE the bank's own flag
Kt['m'] = month_index(Kt['ts'].dt).astype('int16')
gen = test.index[test['y'] == 0]
G = M[M['acct'].isin(gen) & (M['month'] >= month_index(TEST_DATE)) & (M['month'] < month_index(TEST_DATE) + 6)][['acct', 'month', 'big_in', 'n_big_in']]
print(train.groupby('snap').agg(rows=('y', 'size'), mules=('y', 'sum')))
print(f"learning rows {len(train):,} with {int(train['y'].sum())} mule rows ({train[train['y'] == 1].index.nunique()} different mules)")
print(f"test: {len(test):,} accounts, {int(test['y'].sum()):,} mules ({test['y'].mean():.2%})")
print(f"mule money before the bank's flag, Jan-Jun 2025: {len(Kt):,} credits of Rs 1,000+, Rs {Kt['amt'].sum():,.0f}, into {Kt['acct'].nunique():,} mules")
meta = {'train_rows': int(len(train)), 'train_positive_rows': int(train['y'].sum()), 'train_mules': int(train[train['y'] == 1].index.nunique()),
        'test_accounts': int(len(test)), 'test_mules': int(test['y'].sum()), 'test_mules_2026': n_2026,
        'mule_credits': int(len(Kt)), 'mule_money_total': float(Kt['amt'].sum()), 'mules_with_money': int(Kt['acct'].nunique()),
        'top100_money_share': float(Kt.groupby('acct')['amt'].sum().nlargest(100).sum() / Kt['amt'].sum())}
json.dump(meta, open(OUTPUT_DIR + '/meta.json', 'w'), indent=1)

## Step 5. The benchmark rules
- **Same features and default settings** for every method (no extra tuning for anyone).
- **The hold threshold is fixed on the learning data only:** we score each learning row with a model that did not see that account (3 folds, grouped by account), then pick the score that 1 in 100 (or 1 in 1,000) not-yet-flagged accounts exceed. The test data never touches the threshold.
- **The test is scored once.** Ranges in brackets are 95% bootstrap intervals (200 resamples of the test accounts).

**The four methods:** a simple rule with no learning (score = (1 − two-way share) × log(number of contacts)); logistic regression; random forest; gradient boosting (LightGBM).

**The scores**
- **ROC-AUC:** pick one mule and one genuine account at random; how often does the mule score higher? 0.5 = coin toss.
- **PR-AUC:** when the method raises alerts, how many are real mules? A coin toss scores about 0.02 here.
- **Mules caught at 1 false alarm per 100 (or 1,000) genuine accounts**, on 1 Jan 2025.
- **Money held:** in the January–June 2025 replay, the share of mule money (credits of Rs 1,000+ that arrived before the bank's own flag) that would have been held, plus the share of genuine customers who would have had at least one hold for a person to review. Ranges come from resampling the mules 1,000 times, because a few large mules carry much of the money.
- **Money held at equal cost:** methods produce differently scaled scores, so a threshold fixed on the learning data can hold 0.4% of genuine customers for one method and 1.7% for another. To compare methods fairly we also report the money held when **exactly 1 in 100 (or 1 in 1,000) genuine customers** get a hold. Like "mules caught at 1 false alarm per 100", this level is set on the test population, so it is a comparison measure, not a threshold a bank could fix in advance.

In [ ]:
from sklearn.model_selection import GroupKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb

FINAL = BASE + GRAPH + SUSPECT
NO_TWO_WAY = [f for f in FINAL if f not in ('two_way_share', 'contacts_in', 'contacts_out')]
RUNS = [('+ graph + suspect list', FINAL, 'Simple rule'), ('+ graph + suspect list', FINAL, 'Logistic regression'),
        ('+ graph + suspect list', FINAL, 'Random forest'), ('+ graph + suspect list', FINAL, 'LightGBM'),
        ('1-hop only', BASE, 'LightGBM'), ('+ 2-hop graph', BASE + GRAPH, 'LightGBM'),
        ('+ PIN/branch (risky)', FINAL + RISKY, 'LightGBM'), ('final, no two-way information', NO_TWO_WAY, 'LightGBM')]
yt = test['y'].to_numpy(); gen_mask = yt == 0

def make(name):
    if name == 'Logistic regression':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    if name == 'Random forest':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=2, random_state=0)
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, n_jobs=2, verbose=-1, random_state=0)

def prep(X, name, med):
    X = X.astype(float).fillna(med)                                  # gaps filled with the LEARNING median only
    return np.sign(X) * np.log1p(np.abs(X)) if name == 'Logistic regression' else X   # log scale helps the linear model only

def rule(X):
    return ((1 - X['two_way_share'].astype(float).fillna(1)) * np.log1p(X['n_contacts'].astype(float).fillna(0))).to_numpy()

def recall_at(y, s, fpr):
    f, t, _ = roc_curve(y, s); return float(np.interp(fpr, f, t))

def metrics(y, s):
    return {'ROC-AUC': roc_auc_score(y, s), 'PR-AUC': average_precision_score(y, s),
            'caught@1/100': recall_at(y, s, 0.01), 'caught@1/1000': recall_at(y, s, 0.001)}

def bootstrap(y, s, n=200, seed=0):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y)); out.append(metrics(y[i], s[i]))
    d = pd.DataFrame(out)
    return {k: [float(d[k].quantile(0.025)), float(d[k].quantile(0.975))] for k in d}

def month_start(m):
    return pd.Timestamp(year=2020 + int(m) // 12, month=int(m) % 12 + 1, day=1)

MULES = test.index[test['y'] == 1]
MULE_TOTAL = Kt.groupby('acct')['amt'].sum().reindex(MULES).fillna(0).to_numpy()
BOOT = np.random.default_rng(0).integers(0, len(MULES), (1000, len(MULES)))   # 1,000 resamples of the mules

def replay(refresh_scores, th):
    # first month each test account would be put on hold (NaN = never); after that every credit of Rs 1,000+ is held
    first = pd.Series(np.nan, index=test.index)
    for T in REFRESH:
        s, done = refresh_scores[T]
        hit = (s >= th) & ~done & first.isna().to_numpy()              # accounts the bank had already flagged are out of scope
        first[hit] = month_index(T)
    fm = first.reindex(Kt['acct']).to_numpy(); held = ~np.isnan(fm) & (fm <= Kt['m'].to_numpy())
    gm = first.reindex(G['acct']).to_numpy(); gheld = ~np.isnan(gm) & (gm <= G['month'].to_numpy())
    mules = test.index[test['y'] == 1]; started = first[mules].dropna()
    lead = [(lab['flag'][a] - month_start(m)).days for a, m in started.items()]
    per_mule = Kt['amt'][held].groupby(Kt['acct'][held]).sum().reindex(MULES).fillna(0).to_numpy()
    boots = per_mule[BOOT].sum(axis=1) / MULE_TOTAL[BOOT].sum(axis=1)
    return {'money_held': float(Kt['amt'][held].sum() / Kt['amt'].sum()),
            'money_held_range': [float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))],
            'mules_held_before_flag': float(len(started) / len(mules)), 'n_mules_held': int(len(started)),
            'median_days_before_flag': float(np.median(lead)) if lead else None,
            'genuine_ever_held': float(first[test.index[gen_mask]].notna().mean()), 'n_genuine_held': int(first[test.index[gen_mask]].notna().sum()),
            'genuine_money_held': float(G['big_in'][gheld].sum() / G['big_in'].sum())}

## Step 6. Run the benchmark (the longest cell, about 15–20 minutes; the random forest takes most of it)

In [ ]:
rows = []
for set_name, feats, name in RUNS:
    t1 = time.time()
    med = train[feats].astype(float).median()
    if name == 'Simple rule':
        oof = rule(train[feats]); model = None; score = rule
    else:
        Xtr, ytr = prep(train[feats], name, med), train['y'].to_numpy(); oof = np.zeros(len(train))
        for tr_i, va_i in GroupKFold(n_splits=3).split(Xtr, ytr, groups=train.index):   # an account never sits in both halves
            oof[va_i] = make(name).fit(Xtr.iloc[tr_i], ytr[tr_i]).predict_proba(Xtr.iloc[va_i])[:, 1]
        model = make(name).fit(Xtr, ytr)
        score = lambda X, model=model, name=name, med=med: model.predict_proba(prep(X, name, med))[:, 1]
    s_test = score(test[feats])
    row = {'features': set_name, 'method': name, 'n_features': len(feats), **metrics(yt, s_test), 'oof_ROC-AUC': roc_auc_score(train['y'], oof)}
    row['CI'] = bootstrap(yt, s_test)
    rs = {T: (score(refresh[T][feats]), refresh[T]['flagged_by_now'].fillna(False).to_numpy(bool)) for T in REFRESH}
    g = oof[train['y'].to_numpy() == 0]
    for tag, q in [('1/100', 0.99), ('1/1000', 0.999)]:
        th = float(np.quantile(g, q))                                  # fixed on LEARNING data only
        flagged = s_test >= th
        row[tag] = {'threshold': th, 'test_caught': float(flagged[~gen_mask].mean()), 'test_n_caught': int(flagged[~gen_mask].sum()),
                    'test_genuine_flagged': float(flagged[gen_mask].mean()), 'test_n_genuine_flagged': int(flagged[gen_mask].sum()),
                    'test_precision': float(yt[flagged].mean()) if flagged.any() else None,
                    'rural_genuine_flagged': float(flagged[gen_mask & test['rural'].to_numpy()].mean()),
                    'urban_genuine_flagged': float(flagged[gen_mask & ~test['rural'].to_numpy()].mean()),
                    **replay(rs, th)}
    S_ = np.vstack([np.where(done, -np.inf, s) for s, done in rs.values()])   # months x accounts, already-flagged months excluded
    gmax = S_.max(axis=0)[gen_mask]
    for tag, rate in [('1/100', 0.01), ('1/1000', 0.001)]:
        row['equal ' + tag] = replay(rs, float(np.quantile(gmax, 1 - rate)))
    if model is not None:                                              # time to score ONE account, median of 300 tries
        one = prep(test[feats].iloc[:300], name, med); ts = []
        for i in range(len(one)):
            t2 = time.perf_counter(); model.predict_proba(one.iloc[[i]]); ts.append(time.perf_counter() - t2)
        row['ms_per_account'] = float(np.median(ts) * 1000)
    if name == 'LightGBM' and set_name == '+ graph + suspect list':
        imp = pd.Series(model.booster_.feature_importance('gain'), index=feats); imp = imp / imp.sum()
        row['importance'] = imp.sort_values(ascending=False).round(4).to_dict()
    row['seconds'] = round(time.time() - t1)
    rows.append(row)
    print(f"{set_name:30s} {name:20s} ROC-AUC {row['ROC-AUC']:.3f}  money held @1/100: {row['1/100']['money_held']:.1%}  [{row['seconds']} s]")
json.dump(rows, open(OUTPUT_DIR + '/results_detail.json', 'w'), indent=1, default=float)

## Results: the four methods, same features, tested on 2025

In [ ]:
def flat(r):
    a, b = r['1/100'], r['1/1000']
    return {'features': r['features'], 'method': r['method'], 'ROC-AUC': r['ROC-AUC'],
            'ROC-AUC 95% range': f"{r['CI']['ROC-AUC'][0]:.3f}-{r['CI']['ROC-AUC'][1]:.3f}", 'PR-AUC': r['PR-AUC'],
            'caught @1/100': r['caught@1/100'], 'caught @1/1000': r['caught@1/1000'],
            'money held @1/100': a['money_held'], 'money held @1/100 range': '{:.3f}-{:.3f}'.format(*a['money_held_range']),
            'money held @1/1000': b['money_held'],
            'money held, 1 in 100 genuine held': r['equal 1/100']['money_held'],
            'equal-cost range': '{:.3f}-{:.3f}'.format(*r['equal 1/100']['money_held_range']),
            'money held, 1 in 1,000 genuine held': r['equal 1/1000']['money_held'],
            'mules held before flag @1/100': a['mules_held_before_flag'], 'median days before flag @1/100': a['median_days_before_flag'],
            'genuine with a hold @1/100': a['genuine_ever_held'], 'genuine with a hold @1/1000': b['genuine_ever_held'],
            'genuine money held @1/100': a['genuine_money_held'], 'ms per account': r.get('ms_per_account')}
table = pd.DataFrame([flat(r) for r in rows])
table.to_csv(OUTPUT_DIR + '/heimdall_results.csv', index=False)
table[table['features'] == '+ graph + suspect list'].drop(columns='features').round(3)

## Do the 2-hop links and the suspect list help? (LightGBM, same everything else)
- **1-hop only:** the account's own behaviour.
- **+ 2-hop graph:** adds how widely its counterparties and address are shared.
- **+ suspect list:** adds links to accounts the bank had already flagged.
- **+ PIN/branch (risky):** adds the share of flagged accounts in the same PIN code and branch. Kept out of the final model whatever it scores, because it can punish whole neighbourhoods.
- **No two-way information:** in this synthetic data almost every genuine contact is two-way, which may not hold in a real bank. If the score barely drops without it, the model does not depend on that one pattern.

If two rows differ by less than their 95% ranges, treat them as a tie.

In [ ]:
table[table['method'] == 'LightGBM'][['features', 'ROC-AUC', 'ROC-AUC 95% range', 'PR-AUC', 'caught @1/100', 'caught @1/1000',
                              'money held, 1 in 100 genuine held', 'equal-cost range']].round(3)

## The same result, in rupees and people (final LightGBM, threshold = 1 false alarm per 100)

In [ ]:
r = next(r for r in rows if r['method'] == 'LightGBM' and r['features'] == '+ graph + suspect list'); a = r['1/100']
print(f"Mule money that arrived Jan-Jun 2025 before the bank's own flag: Rs {Kt['amt'].sum():,.0f}")
print(f"  held by Heimdall: {a['money_held']:.1%} (95% range {a['money_held_range'][0]:.0%}-{a['money_held_range'][1]:.0%})")
print(f"Mules with a hold before the bank's flag: {a['n_mules_held']} of {int(yt.sum())} ({a['mules_held_before_flag']:.0%}), "
      f"a median of {a['median_days_before_flag']:.0f} days earlier")
print(f"Genuine customers with at least one hold for a person to review: {a['n_genuine_held']:,} of {int(gen_mask.sum()):,} ({a['genuine_ever_held']:.1%})")
print(f"Share of genuine customers' Rs 1,000+ credits held: {a['genuine_money_held']:.2%}")
print(f"On 1 Jan 2025 alone: {a['test_n_caught']} mules flagged and {a['test_n_genuine_flagged']} genuine customers flagged "
      f"({a['test_precision']:.0%} of alerts are real mules)")

## Fairness check
Among genuine customers, how often is each group wrongly flagged at the same threshold (1 Jan 2025)? A large gap means one group would face more wrongful holds, so those flags should go to a person before any hold is placed.

In [ ]:
print(f"Genuine customers wrongly flagged: rural {a['rural_genuine_flagged']:.2%} vs urban {a['urban_genuine_flagged']:.2%} "
      f"(ratio {a['rural_genuine_flagged'] / a['urban_genuine_flagged']:.1f}x)")

## Choosing features without touching 2025: validation inside the learning years
The fair way to decide whether the link features earn their place is to use **only the learning data**: learn from the 2022–2023 snapshots (whose outcomes are known by 1 Jan 2024) and validate on the 1 Jan 2024 snapshot (accounts flagged during 2024). Only about 170 mules are in this validation set, so small differences are noise.

In [ ]:
tr = train[train['snap'] <= pd.Timestamp('2023-01-01')]; va = train[train['snap'] == pd.Timestamp('2024-01-01')]
print(f"learn {len(tr):,} rows ({int(tr['y'].sum())} mule rows) -> validate {len(va):,} accounts ({int(va['y'].sum())} mules)")
val = []
for set_name, feats, name in RUNS + [('1-hop only', BASE, n) for n in ['Simple rule', 'Logistic regression', 'Random forest']]:
    med = tr[feats].astype(float).median()
    if name == 'Simple rule':
        s = rule(va[feats])
    else:
        m_ = make(name).fit(prep(tr[feats], name, med), tr['y'].to_numpy()); s = m_.predict_proba(prep(va[feats], name, med))[:, 1]
    val.append({'features': set_name, 'method': name, **metrics(va['y'].to_numpy(), s)})
val = pd.DataFrame(val)
val.to_csv(OUTPUT_DIR + '/validation_2024.csv', index=False)
val.round(3)

## Why not a random 70/30 split?
Most published results split accounts at random. Here we do that with the **same 2025 accounts and features**: the model learns from 70% of the 2025 mules and is tested on the other 30%, which were flagged in the same months, in the same sweep. Compare with the honest out-of-time result above.

In [ ]:
med = test[FINAL].astype(float).median(); out = []
for tr_i, te_i in StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=42).split(test, yt):
    m_ = make('LightGBM').fit(test[FINAL].iloc[tr_i].astype(float).fillna(med), yt[tr_i])
    out.append(metrics(yt[te_i], m_.predict_proba(test[FINAL].iloc[te_i].astype(float).fillna(med))[:, 1]))
rnd = pd.DataFrame(out).agg(['mean', 'std'])
rnd.to_csv(OUTPUT_DIR + '/random_split_check.csv')
rnd.round(3)

## What the final model relies on (share of LightGBM gain)

In [ ]:
imp = pd.Series(r['importance']).rename('share of gain')
imp.to_csv(OUTPUT_DIR + '/feature_importance.csv')
imp.head(12).round(3)

## What this notebook does NOT show
- **Real banks:** the data is synthetic. Some patterns (for example almost every genuine contact being two-way) may not exist in real banks.
- **Labels are the bank's own flags.** Most mules in this dataset were flagged in one sweep (April–June 2025), and only a few hundred were flagged before 2025, so the models learn from few examples. Undetected mules count as "genuine", so all scores are approximate.
- **Monthly re-scoring:** the money replay re-scores accounts once a month. The real-time check would re-score at every credit, so it could act sooner, and could also raise more alerts.
- **No tuning,** by design, so all methods get equal effort.
- **Speed** here is the time to score one account. The full real-time engine (updating the account's numbers on every transaction, then scoring) is the next stage of the project.